### Phase 6: Generalization

In this phase, the causal inference pipeline is applied across multiple datasets to evaluate its ability to generalize across different domains.

The same pipeline is used for all datasets without introducing artificial normalization or enforcing uniform behavior.

This allows the model to reflect natural variations present in each dataset.

In [1]:
import pandas as pd
import numpy as np
from dowhy import CausalModel
from sklearn.ensemble import RandomForestRegressor

C:\Users\vinee\.conda\envs\ds\Lib\site-packages\sklearn\utils\_param_validation.py:14: UserWarning: A NumPy version >=1.23.5 and <2.5.0 is required for this version of SciPy (detected version 2.5.3)
  from scipy.sparse import csr_matrix, issparse
C:\Users\vinee\.conda\envs\ds\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DATASETS = {
    "education" : {
        "path" : "../data/raw/student-mat.csv",
        "treatment" : "studytime",
        "outcome" : "G3",
        "sep" : ";"
    },
    "healthcare" : {
        "path" : "../data/raw/diabetes.csv",
        "treatment" : "Glucose",
        "outcome" : "Outcome"
    },
    "marketing" : {
        "path" : "../data/raw/Kevin_Hillstrom_MineThatData_E-MailAnalytics_DataMiningChallenge_2008.03.20.csv",
        "treatment" : "segment",
        "outcome" : "conversion"
    }
}

### Multi-Dataset Setup

Three datasets from different domains are used:

- Education dataset (student performance)
- Healthcare dataset (diabetes)
- Marketing dataset (email campaign)

Each dataset has its own treatment and outcome variables, defined based on its structure.

The datasets are processed individually while using a unified pipeline.

In [3]:
def preprocess(df, dataset_name):
    if dataset_name == "marketing":
        df["treatment"] = (df["segment"] != "No E-Mail").astype(int)
        return df, "treatment"
    return df, None

### Dataset-Specific Preprocessing

A preprocessing function is defined to handle differences between datasets.

For example, categorical treatment variables in the marketing dataset are converted into numerical form to ensure compatibility with the model.

No aggressive transformations are applied, and the original data characteristics are preserved.

### Applying Unified Pipeline

The same causal pipeline is applied to each dataset.

For every dataset:
- The causal model is created
- The causal effect is estimated
- Counterfactual analysis is performed
- Bootstrap sampling is used to measure stability

This ensures consistency in evaluation across domains.

In [4]:
def run_pipeline(df, treatment, outcome):
    model = CausalModel(data = df, treatment = treatment, outcome = outcome)
    identified_estimand = model.identify_effect()
    estimate = model.estimate_effect(identified_estimand, method_name = "backdoor.linear_regression")
    sample = df.iloc[0:1].copy()
    cf_sample = sample.copy()
    cf_sample[treatment] += 1
    X = df.drop(columns = [outcome])
    y = df[outcome]
    reg = RandomForestRegressor(random_state = 42)
    reg.fit(X, y)
    original = reg.predict(sample.drop(columns = [outcome]))[0]
    cf = reg.predict(cf_sample.drop(columns = [outcome]))[0]
    cf_effect = cf - original
    effects = []
    for i in range(30):
        sample_df = df.sample(frac = 1, replace = True)
        X = sample_df.drop(columns = [outcome])
        y = sample_df[outcome]
        reg = RandomForestRegressor(random_state = i)
        reg.fit(X, y)
        original = reg.predict(sample.drop(columns = [outcome]))[0]
        cf = reg.predict(cf_sample.drop(columns = [outcome]))[0]
        effects.append(cf - original)
    mean_effect = np.mean(effects)
    std_effect = np.std(effects)
    return {
        "ATE" : estimate.value,
        "CF_effect" : cf_effect,
        "Bootstrap_mean" : mean_effect,
        "Bootstrap_std" : std_effect
    } 

### Causal Effect Estimation

The Average Treatment Effect (ATE) is computed for each dataset using the causal model.

This represents the overall impact of the treatment variable on the outcome.

### Counterfactual Analysis

Counterfactual predictions are generated by modifying the treatment variable and observing the change in predicted outcome.

This provides a data-driven estimate of the treatment effect at the individual level.

### Bootstrap-Based Stability

Bootstrap sampling is used to evaluate how stable the estimated effects are under different sampled datasets.

The mean and standard deviation of the bootstrap results indicate the variability of the model.

In [5]:
results = {}
for name, config in DATASETS.items():
    df = pd.read_csv(config["path"], sep = config.get("sep", ","))
    df, new_treatment = preprocess(df, name)
    df = pd.get_dummies(df, drop_first = True)
    treatment = new_treatment if new_treatment else config["treatment"]
    res = run_pipeline(df, treatment, config["outcome"])
    results[name] = res
results

{'education': {'ATE': np.float64(0.5340011312423609),
  'CF_effect': np.float64(0.009999999999999787),
  'Bootstrap_mean': np.float64(0.008666666666666689),
  'Bootstrap_std': np.float64(0.02061283311165376)},
 'healthcare': {'ATE': np.float64(0.006960225751927673),
  'CF_effect': np.float64(0.0),
  'Bootstrap_mean': np.float64(-0.0030000000000000066),
  'Bootstrap_std': np.float64(0.010049875621120892)},
 'marketing': {'ATE': np.float64(0.0049545711552682475),
  'CF_effect': np.float64(0.0),
  'Bootstrap_mean': np.float64(0.0),
  'Bootstrap_std': np.float64(0.0)}}

In [6]:
results_df = pd.DataFrame(results).T
results_df

,ATE,CF_effect,Bootstrap_mean,Bootstrap_std
education,0.534001,0.01,0.008667,0.020613
healthcare,0.006960,0.00,-0.003000,0.010050
marketing,0.004955,0.00,0.000000,0.000000


### Generalization Results

The results obtained for all datasets show that the causal effects vary across domains.

The education dataset exhibits a relatively strong effect, while healthcare and marketing datasets show very small or negligible effects.

This variation reflects the inherent differences between datasets rather than inconsistencies in the model.

### Interpretation

The results indicate that the pipeline successfully adapts to different datasets while preserving their natural characteristics.

Instead of producing uniform outputs, the model reflects domain-specific behavior.

This confirms that the pipeline generalizes structurally across datasets, even though the magnitude of causal effects differs.

In [7]:
results_df.to_csv("../results/phase6_generalization/generalization_results.csv", index = False)

### Saving Results

The generalization results are stored for use in the evaluation phase.

This allows further analysis of consistency, uncertainty, and robustness.

### Conclusion

This phase demonstrates that the causal inference pipeline can be applied across multiple datasets without modification.

The observed differences in results highlight the importance of domain characteristics, reinforcing the decision to preserve real-world data behavior instead of enforcing artificial consistency.